In [ ]:
import torch
import torch.nn.functional as F

import tonic 
import tonic.transforms as T

import numpy as np
import matplotlib.pyplot as plt

from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cams, plot_combined_frames, plot_activations, combine_polarities
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model

from src.temporal_score_cam import TemporalScoreCAM

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.seed())
# torch.manual_seed(4762538657146009718)

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

In [ ]:
# load the model
train_loader, test_loader = get_dataloaders(
    config["DATASET_NAME"], 
    1, 
    False,
)

event_batches, _ = next(iter(train_loader))
num_channels = event_batches.shape[2]

model = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        num_channels=num_channels,
        time_steps=config["N_TIME_BINS"],
        beta=config["BETA"],
        enable_bntt=config["ENABLE_BNTT"],
    ).to(device)

interactive_load_model(model, config["MODEL_SAVE_DIR"], config["MODEL_NAME"])



In [ ]:
model.eval()
# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model)
conv_layers = [module for module in model.modules() if isinstance(module, torch.nn.Conv2d)]

target_layer = conv_layers[-1] if conv_layers else None
with torch.no_grad():
    for i, (event_batches, labels) in enumerate(test_loader):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        if isinstance(model, tuple):
                # output would be out_spikes
                out_spikes, out_mem = model(event_batches)
                raw_mem = compute_raw_mem(out_spikes, out_mem)
                delta_raw_mem = compute_delta_raw_mem(raw_mem)
                output_class = F.softmax(out_spikes.sum(dim=1), dim=1).cpu().numpy()
                raw_mem = compute_raw_mem(out_spikes, out_mem)
                predicted_label = int(output_class.argmax(axis=1)[0])
                print([f'out_mem: {p:.2f}' for p in out_mem[0, :, predicted_label].cpu().numpy()])
                print([f'out_spikes: {int(p)}' for p in out_spikes[0, :, predicted_label].cpu().numpy()])
        else:
            # output would be out_classes or out_membrane. It will be matched with appropriate criterion
            outputs = model(event_batches)
            delta_outputs = compute_delta_raw_mem(outputs)
            # only take the last time step for classification for the membrane potential-based models
            output_class = F.softmax(outputs[:, -1, :], dim=1).cpu().numpy()
            raw_mem = outputs
            predicted_label = int(output_class.argmax(axis=1)[0])
        delta_raw_mem = compute_delta_raw_mem(raw_mem)

        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\nFrom spike sum {out_spikes.sum(dim=1)}")
        print(f"Predicted label {predicted_label},real label: {labels.item()}\nPredicted gesture: {gesture_map[predicted_label]}, real gesture: {gesture_map[labels.item()]}")
        
        fig, axes = plt.subplots(1, 2, figsize=(13, 6))
        print([f'raw_mem: {p:.2f}' for p in raw_mem[0, :, predicted_label].cpu().numpy()])
        print([f'delta_raw_mem: {p:.2f}' for p in delta_raw_mem[0, :, predicted_label].cpu().numpy()])
        
        y_spikes = out_spikes[0, :, predicted_label].cpu().numpy()
        spike_times = np.nonzero(y_spikes)[0]
        axes[0].plot(range(raw_mem.shape[1]), raw_mem[0, :, predicted_label].cpu().numpy(), label='Raw Mem')
        axes[0].plot(range(out_mem.shape[1]), out_mem[0, :, predicted_label].cpu().numpy(), label='Out Mem', linestyle='--')
        axes[0].eventplot(spike_times, lineoffsets=0, linelengths=2, colors='red', label='Out Spikes')
        axes[0].set_title("Raw Membrane Potential for Predicted Class")
        axes[0].legend()
        axes[0].set_xticks(range(raw_mem.shape[1]))
        axes[0].grid(True)
        axes[0].set_ylim(0, 5)  # Adjust y-axis limit for better visibility
        # Plot Delta Raw Membrane Potential
        axes[1].plot(range(delta_raw_mem.shape[1]), delta_raw_mem[0, :, predicted_label].cpu().numpy(), label='Delta Raw Mem')
        axes[1].plot(range(out_mem.shape[1]), out_mem[0, :, predicted_label].cpu().numpy(), label='Out Mem', linestyle='--')
        axes[1].eventplot(spike_times, lineoffsets=0.5, linelengths=1, colors='red', label='Out Spikes')        
        axes[1].set_title("Delta Raw Membrane Potential for Predicted Class")
        axes[1].legend()
        axes[1].set_xticks(range(delta_raw_mem.shape[1]))
        axes[1].grid(True)
        axes[1].set_ylim(-2, 3)

        single_input = event_batches[0]
        single_input_singleton_batch = single_input.unsqueeze(0)  # Add batch dimension for processing
        combined_frames = combine_polarities(single_input.cpu().numpy())
        plot_frames(single_input.cpu().numpy())
        plot_combined_frames(combined_frames)
        
        # Step through the scorecam parts to ensure everything works
        activations = score_cam.extract_activations(single_input_singleton_batch, target_layer) # (BATCH, TIME, NUM_CHANNELS, HEIGHT, WIDTH)
        upsampled_activations = score_cam.upsample_activations(activations, single_input_singleton_batch, normalize=True)
        print("Activations before normalisation:")
        plot_activations(activations)
        print("Normalised upsampled activations:")
        plot_activations(upsampled_activations)

        # Seems to crash the tunnel connection...        
        # print(f"Individual channel activations (Total {activations.shape[2]} channels):")
        # for i in range(activations.shape[2]):
        #     plot_activations(activations[:, :, i:i+1, :, :], timesteps=5)


        # print(f"Individual upsampled normalized channel activations (Total {upsampled_activations.shape[2]} channels):")
        # for i in range(upsampled_activations.shape[2]):
        #     plot_activations(upsampled_activations[:, :, i:i+1, :, :], timesteps=5)


        # only one sample for now
        break